In [ ]:
import pandas as pd
import numpy as np
import os
import sys

from model_metrics import summarize_model_performance
from model_tuner import loadObjects

sys.path.append("../")

from core.functions import create_shap_plots

import model_metrics

print(f"Model Metrics version: {model_metrics.__version__}")

In [ ]:
model_linear_reg = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/518965103273984050/e23ecdca668749e2a2e565ff3638cf6d/artifacts/lr_log_fatalities/model.pkl"
)
model_lasso_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/518965103273984050/25ca5c790da74777acff4e370ca1f451/artifacts/lasso_log_fatalities/model.pkl"
)
model_xgb_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/518965103273984050/d3892751867d47cb8ac82cbbfc9316dc/artifacts/xgb_log_fatalities/model.pkl"
)
model_cat_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/518965103273984050/d75b126f713f4271a0c5f571c76a9da1/artifacts/cat_log_fatalities/model.pkl"
)

In [ ]:
[col for col in model_xgb_rfe.get_feature_names() if "ash" in col]

In [ ]:
X_train = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_train.parquet"
)

In [ ]:
X_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

In [ ]:
y_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_test_log_fatalities.parquet"
)

In [ ]:
from pathlib import Path
from sklearn.pipeline import Pipeline
import scipy.sparse as sp
import pickle, shap, numpy as np, pandas as pd

DATA_DIR = Path("../data")


def get_shap_beeswarm_full(model_wrapper, X_test):
    fitted_pipeline = model_wrapper.estimator
    estimator = fitted_pipeline.steps[-1][1]
    estimator_type = type(estimator).__name__

    prep = Pipeline(fitted_pipeline.steps[:-1])
    X_te = prep.transform(X_test)
    if sp.issparse(X_te):
        X_te = X_te.toarray()

    tree_based = any(
        n in estimator_type.lower()
        for n in ["xgb", "catboost", "gradientboosting", "randomforest", "tree"]
    )

    if tree_based:
        explainer = shap.TreeExplainer(estimator)
        X_sample = X_te  # use ALL test data — TreeExplainer is fast
    else:
        bg = shap.kmeans(
            (
                prep.transform(X_train).toarray()
                if sp.issparse(prep.transform(X_train))
                else prep.transform(X_train)
            ),
            20,
        )
        explainer = shap.KernelExplainer(estimator.predict, bg)
        idx = np.random.choice(
            X_te.shape[0], size=min(300, X_te.shape[0]), replace=False
        )
        X_sample = X_te[idx]

    shap_values = explainer.shap_values(X_sample)

    try:
        feature_names = fitted_pipeline[:-1].get_feature_names_out()
    except:
        feature_names = [f"feature_{i}" for i in range(X_sample.shape[1])]

    X_sample_df = pd.DataFrame(X_sample, columns=feature_names)
    return shap_values, X_sample_df


for key, model_wrapper in [
    ("lr", model_linear_reg),
    ("lasso", model_lasso_rfe),
    ("xgb", model_xgb_rfe),
    ("catboost", model_cat_rfe),
]:
    print(f"\nRegenerating beeswarm for {key}...")
    sv, X_df = get_shap_beeswarm_full(model_wrapper, X_test)
    out = DATA_DIR / f"shap_beeswarm_{key}.pkl"
    with open(out, "wb") as f:
        pickle.dump({"shap_values": sv, "X": X_df}, f)
    print(f"Saved {out}  shape={sv.shape}")

In [ ]:
df_predictions = pd.DataFrame(
    {
        "y_true_log": y_test.values.ravel(),
        "lr_log": model_linear_reg.estimator.predict(X_test).ravel(),
        "lasso_log": model_lasso_rfe.estimator.predict(X_test).ravel(),
        "xgb_log": model_xgb_rfe.estimator.predict(X_test).ravel(),
        "catboost_log": model_cat_rfe.estimator.predict(X_test).ravel(),
        "split": "test",
    }
)

df_predictions.to_csv("../data/predictions.csv", index=False)

print(df_predictions.shape)
print(df_predictions.head())

In [ ]:
pre = model_linear_reg.estimator.named_steps[
    "preprocess_column_transformer_Preprocessor"
]

feature_names = pre.get_feature_names_out()

In [ ]:
regression_metrics = summarize_model_performance(
    model=[
        model_linear_reg,
        model_lasso_rfe,
        model_xgb_rfe,
        model_cat_rfe,
    ],
    model_title=[
        "Linear Regression",
        "Lasso RFE",
        "XGBoost RFE",
        "CatBoost RFE",
    ],
    X=X_test,
    y=y_test,
    model_type="regression",
    include_adjusted_r2=True,
    return_df=True,
    decimal_places=2,
    overall_only=False,
)

In [ ]:
regression_metrics = regression_metrics.drop(
    columns=["Metric", "Variable", "Coefficient"]
)

In [ ]:
regression_metrics

In [ ]:
len(model_linear_reg.get_feature_names())

In [ ]:
len(model_xgb_rfe.get_feature_names())

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": model_linear_reg.get_feature_names(),
        "coef": model_linear_reg.estimator.named_steps["lr"].coef_,
    }
).sort_values("coef", key=abs, ascending=False)

coef_df.head(20)

In [ ]:
model_xgb_rfe.estimator

In [ ]:
df = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/raw/acled_ukraine_data_2026_01_02.parquet"
)

In [ ]:
X_test = X_test.assign(
    **pd.get_dummies(X_test[["sub_event_type", "source_scale"]], dtype=int)
)

In [ ]:
[col for col in X_test.columns if "Regional" in col]

In [ ]:
[col for col in X_test.columns if "source" in col]

In [ ]:
[col for col in model_xgb_rfe.get_feature_names() if "source" in col]

In [ ]:
import os

os.getcwd()

In [ ]:
os.path.exists("../data")

In [ ]:
label_map = (None,)
apply_label_map = (True,)


label_replacements = {
    # Sub Event Types
    "Shelling/artillery/missile attack": "Shelling / Missile",
    "Peaceful protest": "Peaceful protest",
    "Mob violence": "Mob violence",
    "Government regains territory": "Gov regains territory",
    "Change to group/activity": "Group change",
    "Armed clash": "Armed clash",
    "Abduction/forced disappearance": "Forced disappearance",
    # Source Scale
    "Subnational-Regional": "Subnational Regional",
    "Subnational": "Subnational",
    "Other-Subnational": "Other Subnational",
    "Other-National": "Other National",
    "National": "National",
    "Other-International": "Other International",
    "International": "International",
    "New media-National": "New media National",
    "New media-International": "New media International",
}

In [ ]:
source_scale_map = {
    "International": "Intl",
    "Local partner-New media": "Local+Media",
    "National": "National",
    "National-International": "Nat+Intl",
    "National-Regional": "Nat+Reg",
    "New media": "Media",
    "New media-National": "Media+Nat",
    "Other": "Other",
    "Other-International": "Other+Intl",
    "Other-National": "Other+Nat",
    "Other-New media": "Other+Media",
    "Other-Regional": "Other+Reg",
    "Other-Subnational": "Other+Subnat",
    "Regional": "Regional",
    "Subnational": "Subnat",
    "Subnational-National": "Subnat+Nat",
    "Subnational-Regional": "Subnat+Reg",
}

event_map = {
    "Abduction/forced disappearance": "Abduction",
    "Agreement": "Agreement",
    "Air/drone strike": "Airstrike",
    "Armed clash": "Armed Clash",
    "Arrests": "Arrests",
    "Attack": "Attack",
    "Change to group/activity": "Group Change",
    "Chemical weapon": "Chemical",
    "Disrupted weapons use": "Disrupted Use",
    "Government regains territory": "Govt Gains",
    "Grenade": "Grenade",
    "Looting/property destruction": "Looting",
    "Mob violence": "Mob Violence",
    "Non-state actor overtakes territory": "NSA Gains",
    "Other": "Other",
    "Peaceful protest": "Peaceful Protest",
    "Remote explosive/landmine/IED": "IED",
    "Sexual violence": "Sexual Violence",
    "Shelling/artillery/missile attack": "Shelling",
    "Suicide bomb": "Suicide Bomb",
    "Violent demonstration": "Violent Protest",
}

In [ ]:
# from model_metrics import plot_3d_pdp

# plot_3d_pdp(
#     model=model_xgb_rfe.estimator,
#     dataframe=X_test,
#     feature_names=["sub_event_type", "source_scale"],
#     x_label="",
#     y_label="",
#     x_label_map=event_map,
#     y_label_map=source_scale_map,
#     matplotlib_colormap="viridis",
#     z_label="Partial Dependence",
#     title="3D Partial Dependence Plot of Event Type vs. Source Scale",
#     html_file_path="../data",
#     # image_filename="3d_pdp",
#     html_file_name="3d_pdp.html",
#     image_path_svg="../data",
#     plot_type="static",
#     text_wrap=80,
#     zoom_out_factor=1.3,
#     grid_resolution=35,
#     label_fontsize=8,
#     tick_fontsize=6,
#     title_x=0.38,
#     figsize=(15, 8),
#     top_margin=20,
#     # bottom_margin=120,
#     right_margin=80,
#     view_angle=(23, 50),
#     left_margin=70,
#     cbar_x=0.9,
#     cbar_thickness=25,
#     show_modebar=False,
#     enable_zoom=True,
#     save_plots="html",
# )

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=model_xgb_rfe.estimator,
    dataframe=X_test,
    feature_names=["sub_event_type", "source_scale"],
    x_label="",
    y_label="",
    # x_label="sub_event_type",
    # y_label="source_scale",
    x_label_map=event_map,
    y_label_map=source_scale_map,
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="interactive",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=25,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=True,
    enable_zoom=True,
    save_plots="html",
    modebar_image_format="svg",
)

In [ ]:
from model_metrics import summarize_model_performance

In [ ]:
y_test.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb_rfe.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")

In [ ]:
log_preds["pred_fatalities"] = round(
    np.expm1(log_preds["log_pred_fatalities"]), 0
).astype(int)

In [ ]:
log_preds_merge = log_preds.join(X_test, how="inner", on="event_id_cnty")

In [ ]:
y_test_actual = pd.DataFrame(
    np.expm1(y_test["actual_log_fatalities"]).round(0).astype(int),
    columns=["actual_log_fatalities"],
).rename(columns={"actual_log_fatalities": "actual_fatalities"})

In [ ]:
y_test_actual

In [ ]:
log_preds_merge = y_test_actual.merge(log_preds_merge, how="inner", on="event_id_cnty")

In [ ]:
log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

In [ ]:
model_xgb_rfe.get_feature_names()

In [ ]:
dir(model_xgb_rfe)